In [1]:
import argparse
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## AE (Autoencoder)

`AE` は入力画像を 64 次元の潜在表現 `z` に圧縮し、そこから元画像を再構成する最も基本的な自己符号化器です。

- `__init__`: エンコーダ `enc` とデコーダ `dec` を定義します。MNIST の 28x28 画像を 784 次元ベクトルとして扱います。
- `forward`: 入力 `x` を平坦化して `enc` に通し、潜在変数 `z` を得てから `dec` で再構成画像 `x_hat` を出力します。戻り値は `(x_hat, z)` です。


In [2]:
class AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(28 * 28, 256), nn.ReLU(), nn.Linear(256, 64))
        self.dec = nn.Sequential(
            nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 28 * 28), nn.Sigmoid()
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        z = self.enc(x)
        x_hat = self.dec(z)
        return x_hat, z


## VAE (Variational Autoencoder)

`VAE` は潜在変数を確率分布として扱う自己符号化器です。平均 `mu` と対数分散 `logvar` を学習し、そこからサンプリングした `z` を使って再構成します。

- `__init__`: エンコーダ側の全結合層と、平均・分散用の出力層、デコーダ側の層を定義します。
- `encode`: 入力 `x` から潜在分布のパラメータ `(mu, logvar)` を計算します。
- `reparam`: 再パラメータ化トリックを使って、勾配を流せる形で `z = mu + eps * std` を生成します。
- `decode`: 潜在変数 `z` から再構成画像 `x_hat` を生成します。
- `forward`: `encode -> reparam -> decode` をまとめて実行し、戻り値として `(x_hat, mu, logvar)` を返します。


In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc_mu = nn.Linear(256, 64)
        self.fc_logvar = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 256)
        self.fc3 = nn.Linear(256, 28 * 28)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.fc2(z))
        return torch.sigmoid(self.fc3(h))

    def forward(self, x):
        x = x.view(x.size(0), -1)
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar

## VectorQuantizer / VQVAE

`VectorQuantizer` は連続値の潜在表現を、コードブック中の最も近い離散ベクトルに置き換える部品です。`VQVAE` はこの量子化器を挟んだ自己符号化器です。

### VectorQuantizer
- `__init__`: コードブックのサイズ `K`、埋め込み次元 `D`、損失係数 `beta` を設定します。
- `forward`: 潜在表現 `z` とコードブック各ベクトルの距離を計算し、最も近い埋め込みを `z_q` として選びます。量子化損失も同時に返し、戻り値は `(z_q, loss)` です。

### VQVAE
- `__init__`: エンコーダ、`VectorQuantizer`、デコーダを組み合わせてモデル全体を構成します。
- `forward`: 入力 `x` をエンコードし、潜在表現を量子化してから再構成します。戻り値は `(x_hat, vq_loss)` です。


In [ ]:

class VectorQuantizer(nn.Module):
    def __init__(self, K=512, D=64, beta=0.25):
        super().__init__()
        self.embedding = nn.Embedding(K, D)
        self.embedding.weight.data.uniform_(-1 / K, 1 / K)
        self.beta = beta

    def forward(self, z):
        flat = z.view(-1, z.size(-1))

        dist = (
            flat.pow(2).sum(1, keepdim=True)
            - 2 * flat @ self.embedding.weight.t()
            + self.embedding.weight.pow(2).sum(1)
        )

        idx = torch.argmin(dist, dim=1)
        z_q = self.embedding(idx).view_as(z)

        loss = F.mse_loss(z_q.detach(), z) + self.beta * F.mse_loss(z_q, z.detach())
        z_q = z + (z_q - z).detach()

        return z_q, loss


class VQVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(28 * 28, 256), nn.ReLU(), nn.Linear(256, 64))
        self.vq = VectorQuantizer()
        self.dec = nn.Sequential(
            nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 28 * 28), nn.Sigmoid()
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        z = self.enc(x)
        z_q, vq_loss = self.vq(z)
        x_hat = self.dec(z_q)
        return x_hat, vq_loss

## train 関数

`train(model, loader, device, model_type)` は学習ループ全体を担当します。`model_type` に応じて損失関数を切り替え、3 エポック分の最適化を実行します。

- `ae`: 再構成誤差として MSE のみを使います。
- `vae`: 再構成誤差 `recon` に加えて、潜在分布を正則化する KL ダイバージェンス `kl` を足します。
- `vqvae`: 再構成誤差 `recon` に加えて、量子化器が返す `vq_loss` を足します。
- 各ミニバッチで `zero_grad -> backward -> step` を行い、各エポック末尾で最後の loss を表示します。


In [3]:
def train(model, loader, device, model_type):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(3):
        for x, _ in loader:
            x = x.to(device)

            if model_type == "ae":
                x_hat, _ = model(x)
                loss = F.mse_loss(x_hat, x.view(x.size(0), -1))

            elif model_type == "vae":
                x_hat, mu, logvar = model(x)
                recon = F.mse_loss(x_hat, x.view(x.size(0), -1))
                kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                loss = recon + kl

            elif model_type == "vqvae":
                x_hat, vq_loss = model(x)
                recon = F.mse_loss(x_hat, x.view(x.size(0), -1))
                loss = recon + vq_loss

            opt.zero_grad()
            loss.backward()
            opt.step()

        print(f"epoch {epoch} loss {loss.item():.4f}")


## 再構成結果の可視化

`show_reconstructions(model, loader, device, model_type)` は学習済みモデルに数枚の画像を通し、元画像と再構成画像を上下 2 段で並べて表示します。

- 上段に元画像、下段に再構成画像を表示します。
- `ae` / `vae` / `vqvae` の返り値の違いを内部で吸収して、同じ描画関数で扱います。


In [ ]:
def reconstruct_batch(model, x, model_type):
    if model_type == "ae":
        x_hat, _ = model(x)
    elif model_type == "vae":
        x_hat, _, _ = model(x)
    elif model_type == "vqvae":
        x_hat, _ = model(x)
    else:
        raise ValueError(f"unknown model_type: {model_type}")

    return x_hat


def show_reconstructions(model, loader, device, model_type, num_images=8):
    was_training = model.training
    model.eval()

    x, _ = next(iter(loader))
    x = x[:num_images].to(device)

    with torch.no_grad():
        x_hat = reconstruct_batch(model, x, model_type)

    originals = x.detach().cpu().view(-1, 28, 28)
    recons = x_hat.detach().cpu().view(-1, 28, 28)

    fig, axes = plt.subplots(2, num_images, figsize=(1.6 * num_images, 3.5), squeeze=False)

    for i in range(num_images):
        axes[0, i].imshow(originals[i], cmap="gray")
        axes[0, i].set_title(f"orig {i}")
        axes[1, i].imshow(recons[i], cmap="gray")
        axes[1, i].set_title(f"recon {i}")

        axes[0, i].axis("off")
        axes[1, i].axis("off")

    plt.tight_layout()
    plt.show()

    if was_training:
        model.train()


## main 関数

`main(model_type="ae")` は実行準備をまとめる入口です。使用デバイスを決め、MNIST データセットを読み込み、指定したモデルを生成して `train` を呼び出します。

- `device`: CUDA が使えれば GPU、使えなければ CPU を選びます。
- `ds`, `loader`: MNIST の学習データと DataLoader を準備します。
- `model_type`: `ae` / `vae` / `vqvae` に応じてモデルを切り替えます。
- 学習後に `show_reconstructions(...)` を呼び、元画像と再構成画像を表示します。
- 最後の `main("ae")` で、ノートブック実行時は AE を学習して可視化する設定になっています。


In [4]:
def main(model_type="ae"):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    ds = datasets.MNIST(
        "./data", train=True, download=True, transform=transforms.ToTensor()
    )
    loader = DataLoader(ds, batch_size=128, shuffle=True)

    if model_type == "ae":
        model = AE().to(device)
    elif model_type == "vae":
        model = VAE().to(device)
    elif model_type == "vqvae":
        model = VQVAE().to(device)

    train(model, loader, device, model_type)
    show_reconstructions(model, loader, device, model_type)
    return model, loader


# 実行
main("ae")


epoch 0 loss 0.0182
epoch 1 loss 0.0124
epoch 2 loss 0.0091
